In [2]:
import requests
from bs4 import BeautifulSoup

In [3]:
page = requests.get("https://pamyat-naroda.ru/")

In [3]:
page

<Response [403]>

In [3]:
from selenium import webdriver

In [4]:
driver = webdriver.Chrome()

In [5]:
search_result_1stpage = "https://pamyat-naroda.ru/heroes/?adv_search=y&last_name=%D0%9F%D1%83%D1%88%D0%BA%D0%B8%D0%BD&first_name=&middle_name=&date_birth_from=&static_hash=a27d50051f8df3b68a842d0c89cc4dd1b3573f3600cdbc1aa8742bd494516397v9&group=all&types=same_doroga&page=1&grouppersons=1"

In [7]:
driver.get(search_result_1stpage)

In [8]:
text = driver.page_source
soup = BeautifulSoup(text)
soup

<html data-user-id="" data-user-update="" lang="ru&gt;
&lt;head data-static-hash="><head><style>@charset "UTF-8";[ng\:cloak],[ng-cloak],[data-ng-cloak],[x-ng-cloak],.ng-cloak,.x-ng-cloak,.ng-hide:not(.ng-hide-animate){display:none !important;}ng\:form{display:block;}.ng-animate-shim{visibility:hidden;}.ng-anchor{position:absolute;}</style><meta charset="utf-8"/>
<meta content="IE=edge" http-equiv="X-UA-Compatible" id="iexua"/>
<meta content="width=device-width, initial-scale=1.0, minimum-scale=1.0, maximum-scale=1.0" name="viewport"/>
<meta content="GHa1TWL6CUWtSiWiXI6Rugb0_UCTrD3Vpe830aFfoZ" name="_globalsign-domain-verification"/>
<!--[if IE]>
    <meta http-equiv="cache-control" content="max-age=0"/>
    <meta http-equiv="cache-control" content="no-cache"/>
    <meta http-equiv="expires" content="0"/>
    <meta http-equiv="expires" content="Tue, 01 Jan 1980 1:00:00 GMT"/>
    <meta http-equiv="pragma" content="no-cache"/>
    <![endif]-->
<title>Память народа::Поиск документов по ге

In [10]:
pagination = str(soup.find_all("div", class_="pagination-wrap"))
num_pages = int(pagination[pagination.rfind("(") + 1: pagination.rfind(")")])
num_pages

67

In [12]:
pages = [f"https://pamyat-naroda.ru/heroes/?adv_search=y&last_name=%D0%9F%D1%83%D1%88%D0%BA%D0%B8%D0%BD&first_name=&middle_name=&date_birth_from=&static_hash=a27d50051f8df3b68a842d0c89cc4dd1b3573f3600cdbc1aa8742bd494516397v9&group=all&types=same_doroga&page={i}&grouppersons=1" for i in range(1, num_pages + 1)]

In [13]:
import json

with open("person_links.json") as f:
    person_links = json.load(f)

In [17]:
person_dict = dict()
for i in range(len(person_links)):
    person_dict[i + 1] = person_links[i]

In [18]:
with open("person_dict.json", "w") as f:
    json.dump(person_dict, f, indent=4)

In [13]:
from tqdm import tqdm

person_links = []
for page in tqdm(pages):
    driver.get(page)
    text = driver.page_source
    soup = BeautifulSoup(text)
    persons = soup.find_all("a", class_="heroes-list-item-name")
    for person in persons:
        person_links.append(person["href"])

100%|██████████████████████████████████████████████████████████████████████████████████| 67/67 [06:35<00:00,  5.90s/it]


In [15]:
import json

with open("person_links.json", "w") as f:
    json.dump(person_links, f, indent=4)

In [17]:
len(person_links)

230

In [9]:
import json

with open("person_dict.json") as f:
    person_dict = json.load(f)

In [18]:
from tqdm import tqdm

res = dict()
links = list(person_dict.values())
for i in tqdm(range(len(links))):
    link = links[i]
    driver.get(link)
    text = driver.page_source
    soup = BeautifulSoup(text)
    title = soup.title.text
    title = title[:title.rfind(" :")]
    photo_links = []
    try:
        photo_links = [soup.find("a", class_="hero-card-person-photo__item fancybox active js-hero-person-card-photo")["href"]]
        element = driver.find_element(By.XPATH, '/html/body/div[1]/div[4]/div[1]/div[2]/div[1]/div[2]/div/div[2]/div[1]/div[1]/div[1]/div/div/div[1]/a[1]')
        #element.screenshot(f"images/{i}.png")
    except:
        pass
    
    info = list(soup.find_all("div", class_="heroes_card__doc-box"))
    res[link] = {"name": title, "links": photo_links, "information": str(info), "image_file": f"images/{i}.png"}
    
with open("res_dict_1.json", "w", encoding="utf-8") as f:
    json.dump(res, f, indent=4, ensure_ascii=False)

100%|██████████████████████████████████████████████████████████████████████████████| 668/668 [1:01:36<00:00,  5.53s/it]


In [17]:
res

{'https://pamyat-naroda.ru/heroes/person-hero121494310/?backurl=%2Fheroes%2F%3Fadv_search%3Dy%26last_name%3D%D0%9F%D1%83%D1%88%D0%BA%D0%B8%D0%BD%26first_name%3D%26middle_name%3D%26date_birth_from%3D%26static_hash%3Da27d50051f8df3b68a842d0c89cc4dd1b3573f3600cdbc1aa8742bd494516397v9%26group%3Dall%26types%3Dsame_doroga%26page%3D1%26grouppersons%3D1&': {'name': 'Пушкин Михаил Константинович',
  'links': ['https://cdn.pamyat-naroda.ru/images/roadheroes%2Fc0%2Fc0bc2f873c03191e9e2205b2ec84cca5_crop.jpg'],
  'information': '[<div class="heroes_card__doc-box">\n<div class="hero-card-docs-item__info" data-dict=" ">\n<b>Дата рождения:</b>\n                                                                                                                                                \n                                                                    __.__.1925\n                                                            \n                        </div>\n<div class="hero-card-docs-item__info" data

In [1]:
from copy import deepcopy

res_1 = deepcopy(res)
for k, v in res_1.items():
    res_1["information"] = str(res_1["information"])

NameError: name 'res' is not defined

In [14]:
with open("res_dict.json", "w", encoding="utf-8") as f:
    json.dump(res, f, indent=4, ensure_ascii=False)

In [21]:
for el in res:
    print(el["information"].find("div", class_="hero-card-docs-item__info"))

<div class="hero-card-docs-item__info" data-dict=" ">
<b>Дата рождения:</b>
                                                                                                                                                
                                                                    __.__.1925
                                                            
                        </div>
<div class="hero-card-docs-item__info" data-dict=" ">
<b>Дата рождения:</b>
                                                                                                                                                
                                                                    07.11.1919
                                                            
                        </div>
<div class="hero-card-docs-item__info" data-dict=" ">
<b>Дата рождения:</b>
                                                                                                                                            

AttributeError: 'NoneType' object has no attribute 'find'

In [22]:
el

{'name': 'Пушкин Василий Никитович',
 'links': ['https://cdn.pamyat-naroda.ru/images/roadheroes%2F1d%2F1dbead5f1353efc835602c97d377739c_crop.jpg'],
 'information': None,
 'image_file': 'images/3.png'}

In [2]:
from copy import deepcopy

res_1 = deepcopy(res)

NameError: name 'res' is not defined

In [13]:
driver.get(person_links[0])

In [14]:
driver.page_source

'<html lang="ru>\n<head data-static-hash=" b3573f3600cdbc1aa8742bd494516397v9"="" data-user-id="" data-user-update=""><head data-user-update=""><style>@charset "UTF-8";[ng\\:cloak],[ng-cloak],[data-ng-cloak],[x-ng-cloak],.ng-cloak,.x-ng-cloak,.ng-hide:not(.ng-hide-animate){display:none !important;}ng\\:form{display:block;}.ng-animate-shim{visibility:hidden;}.ng-anchor{position:absolute;}</style><meta charset="UTF-8">\n    <meta http-equiv="X-UA-Compatible" content="IE=edge" id="iexua">\n    <meta name="viewport" content="width=device-width, initial-scale=1.0, minimum-scale=1.0, maximum-scale=1.0">\n    <meta name="_globalsign-domain-verification" content="GHa1TWL6CUWtSiWiXI6Rugb0_UCTrD3Vpe830aFfoZ">\n    <!--[if IE]>\n    <meta http-equiv="cache-control" content="max-age=0"/>\n    <meta http-equiv="cache-control" content="no-cache"/>\n    <meta http-equiv="expires" content="0"/>\n    <meta http-equiv="expires" content="Tue, 01 Jan 1980 1:00:00 GMT"/>\n    <meta http-equiv="pragma" cont

In [16]:
from selenium.webdriver.common.by import By

element = driver.find_element(By.XPATH, '/html/body/div[1]/div[4]/div[1]/div[2]/div[1]/div[2]/div/div[2]/div[1]/div[1]/div[1]/div/div/div[1]/a[1]')
element.screenshot("element_screenshot.png")

True

In [67]:
res = []
for link in tqdm(person_links):
    driver.get(link)
    text = driver.page_source
    soup = BeautifulSoup(text)
    title = soup.title.text
    title = title[:title.rfind(" :")]
    photo_links = []
    try:
        photo_links = [soup.find("a", class_="hero-card-person-photo__item fancybox active js-hero-person-card-photo")["href"]]
    except:
        pass
    try:
        for el in soup.find_all("a", class_="hero-card-person-photo__item fancybox js-hero-person-card-photo"):
            photo_links.append(el["href"])
    except:
        pass
    info = soup.find("div", class_="js-hero-card-doc hero-card-docs-item hero-card-docs-item_spisok")
    res.append({"name": title, "links": photo_links, "information": info})

100%|████████████████████████████████████████████████████████████████████████████████| 668/668 [58:30<00:00,  5.26s/it]


In [69]:
len([el for el in res if "Пушкин" in el["name"]])

163

In [70]:
exact_surname = [el for el in res if "Пушкин" in el["name"]]

In [81]:
proxies = {
  "http": None,
  "https": None,
}

requests.get(exact_surname[0]["links"][0], proxies=proxies).content

b'<html>\r\n<head><title>503 Service Temporarily Unavailable</title></head>\r\n<body>\r\n<center><h1>503 Service Temporarily Unavailable</h1></center>\r\n<hr><center>nginx</center>\r\n</body>\r\n</html>\r\n'

In [82]:
url = exact_surname[0]["links"][0]

In [88]:
url

'https://cdn.pamyat-naroda.ru/images/roadheroes%2Fc0%2Fc0bc2f873c03191e9e2205b2ec84cca5_crop.jpg'

In [84]:
driver.page_source

'<html><head><title>503 Service Temporarily Unavailable</title></head>\n<body>\n<center><h1>503 Service Temporarily Unavailable</h1></center>\n<hr><center>nginx</center>\n\n\n\n\n\n\n\n\n</body></html>'

In [86]:
!pip install cloudscraper

                                              0.0/99.7 kB ? eta -:--:--
                                              0.0/99.7 kB ? eta -:--:--
                                              0.0/99.7 kB ? eta -:--:--
     ------------                             30.7/99.7 kB 1.3 MB/s eta 0:00:01
     ------------                             30.7/99.7 kB 1.3 MB/s eta 0:00:01
     ------------                             30.7/99.7 kB 1.3 MB/s eta 0:00:01
     -------------------------------        81.9/99.7 kB 508.4 kB/s eta 0:00:01
     -------------------------------        81.9/99.7 kB 508.4 kB/s eta 0:00:01
     -------------------------------        81.9/99.7 kB 508.4 kB/s eta 0:00:01
     -----------------------------------    92.2/99.7 kB 308.0 kB/s eta 0:00:01
     -------------------------------------- 99.7/99.7 kB 271.8 kB/s eta 0:00:00


In [87]:
import cloudscraper

scraper = cloudscraper.create_scraper()

req = scraper.get(url)
print(req.status_code)

503


In [95]:
driver = webdriver.Chrome()

In [106]:
driver.get(exact_surname[0]["links"][0])

In [93]:
exact_surname[0]["links"][0]

'https://cdn.pamyat-naroda.ru/images/roadheroes%2Fc0%2Fc0bc2f873c03191e9e2205b2ec84cca5_crop.jpg'

In [107]:
driver.page_source

'<html><head><title>503 Service Temporarily Unavailable</title></head>\n<body>\n<center><h1>503 Service Temporarily Unavailable</h1></center>\n<hr><center>nginx</center>\n\n\n\n\n\n\n\n\n</body></html>'

In [102]:
sum([len(el["links"]) for el in exact_surname])

221

In [110]:
import json

with open("urls.json", "w") as f:
    json.dump(url_list, f, indent=4)

In [19]:
with open("scraping_res.json", "w") as f:
    json.dump(res, f, indent=4)

TypeError: Object of type Tag is not JSON serializable

In [114]:
res

[{'name': 'Пушкин Михаил Константинович',
  'links': ['https://cdn.pamyat-naroda.ru/images/roadheroes%2Fc0%2Fc0bc2f873c03191e9e2205b2ec84cca5_crop.jpg',
   'https://cdn.pamyat-naroda.ru/images/roadheroes%2F1c%2F1c00106b1117ef69f73c3271902c7ed9_crop.jpg',
   'https://cdn.pamyat-naroda.ru/images/roadheroes%2F36%2F3693284544645c2cf94dd8eac0e49320_crop.jpg'],
  'information': <div class="js-hero-card-doc hero-card-docs-item hero-card-docs-item_spisok" data-box_number="" data-doc-documents-id="" data-doc-id="9205769" data-doc-index="isp" data-doc-pages="[]" data-doc-pages-id="" data-doc-show="N" data-doc-type="chelovek_spisok" data-nomer_dela="1" data-nomer_fonda="6559" data-nomer_opisi="267771с" data-podvig_1="" data-podvig_2="" data-podvig_3="" data-section="ЦАМО" data-shkaf_i_yaschik="" data-shkaf_number="" data-storage_number="">
  <div class="hero-card-docs-item__name">
          Пушкин Михаил Константинович    </div>
  <div class="hero-card-docs-item__reward">
                        

In [1]:
from copy import deepcopy


res_1 = deepcopy(res)
for el in res_1:
    el["information"] = str(el["information"])

NameError: name 'res' is not defined

In [108]:
[el["name"] for el in exact_surname]

['Пушкин Михаил Константинович',
 'Пушкин Александр Андреевич',
 'Пушкин Евгений Иванович',
 'Пушкин Василий Никитович',
 'Пушкин Владимир Федорович',
 'Пушкин Михаил Павлович',
 'Пушкин Василий Андреевич',
 'Пушкин Николай Андреевич',
 'Пушкин Алексей Федорович',
 'Пушкин Леонид Михайлович',
 'Пушкин Петр Александрович',
 'Пушкин Василий Владимирович',
 'Пушкин Александр Федорович',
 'Пушкин Степан Михайлович',
 'Пушкин Петр Иванович',
 'Пушкин Семен Иванович',
 'Пушкин Василий Петрович',
 'Пушкин Андрей Никитич',
 'Пушкин Павел Михайлович',
 'Пушкин Павел Иванович',
 'Пушкин Александр Михайлович',
 'Пушкин Максим Григорьевич',
 'Пушкин Василий Васильевич',
 'Пушкин Ефим Григорьевич',
 'Пушкин Николай Васильевич',
 'Пушкин Федор Антонович',
 'Пушкин Иван Егорович',
 'Пушкин Константин Яковлевич',
 'Пушкин Александр Федотович',
 'Пушкин Дмитрий Ильич',
 'Пушкин Владимир Васильевич',
 'Пушкин Павел Александрович',
 'Пушкин Иван Васильевич',
 'Пушкин Иван Тарасович',
 'Пушкин Андрей Дмит

In [104]:
url_list = []
for el in exact_surname:
    url_list += el["links"]

In [105]:
url_list

['https://cdn.pamyat-naroda.ru/images/roadheroes%2Fc0%2Fc0bc2f873c03191e9e2205b2ec84cca5_crop.jpg',
 'https://cdn.pamyat-naroda.ru/images/roadheroes%2F1c%2F1c00106b1117ef69f73c3271902c7ed9_crop.jpg',
 'https://cdn.pamyat-naroda.ru/images/roadheroes%2F36%2F3693284544645c2cf94dd8eac0e49320_crop.jpg',
 'https://cdn.pamyat-naroda.ru/images/roadheroes%2F10%2F10c7cb40094a17f639f7215d31566ae9_crop.jpg',
 'https://cdn.pamyat-naroda.ru/images/roadheroes%2Fc1%2Fc12a07d8f47fbd6c9a896c76a414f314_crop.jpg',
 'https://cdn.pamyat-naroda.ru/images/roadheroes%2F1d%2F1dbead5f1353efc835602c97d377739c_crop.jpg',
 'https://cdn.pamyat-naroda.ru/images/roadheroes%2Fdf%2Fdf8e003ddbb53649ca8fc83216702fc1_crop.jpg',
 'https://cdn.pamyat-naroda.ru/images/roadheroes%2F04%2F04c94bf9d1124036342ff20d72bb6875_crop.jpg',
 'https://cdn.pamyat-naroda.ru/images/roadheroes%2F56%2F56f1ccbc796f08a302538b7588ae0f5b_crop.jpg',
 'https://cdn.pamyat-naroda.ru/images/Images_554_2019%2FDIP%2F006%2F169%2F169-27%2F08885886%2F000023

In [40]:
driver.get(person_links[0])

In [42]:
text = driver.page_source

In [44]:
soup = BeautifulSoup(text)
soup

<html b3573f3600cdbc1aa8742bd494516397v9="" data-user-id="" data-user-update="" lang="ru&gt;
&lt;head data-static-hash="><head data-user-update=""><style>@charset "UTF-8";[ng\:cloak],[ng-cloak],[data-ng-cloak],[x-ng-cloak],.ng-cloak,.x-ng-cloak,.ng-hide:not(.ng-hide-animate){display:none !important;}ng\:form{display:block;}.ng-animate-shim{visibility:hidden;}.ng-anchor{position:absolute;}</style><meta charset="utf-8"/>
<meta content="IE=edge" http-equiv="X-UA-Compatible" id="iexua"/>
<meta content="width=device-width, initial-scale=1.0, minimum-scale=1.0, maximum-scale=1.0" name="viewport"/>
<meta content="GHa1TWL6CUWtSiWiXI6Rugb0_UCTrD3Vpe830aFfoZ" name="_globalsign-domain-verification"/>
<!--[if IE]>
    <meta http-equiv="cache-control" content="max-age=0"/>
    <meta http-equiv="cache-control" content="no-cache"/>
    <meta http-equiv="expires" content="0"/>
    <meta http-equiv="expires" content="Tue, 01 Jan 1980 1:00:00 GMT"/>
    <meta http-equiv="pragma" content="no-cache"/>
   

In [54]:
soup.title.text

'Пушкин Михаил Константинович :: Память народа'

In [55]:
soup.find("a", class_="hero-card-person-photo__item fancybox active js-hero-person-card-photo")["href"]

'https://cdn.pamyat-naroda.ru/images/roadheroes%2Fc0%2Fc0bc2f873c03191e9e2205b2ec84cca5_crop.jpg'

In [49]:
for el in soup.find_all("a", class_="hero-card-person-photo__item fancybox js-hero-person-card-photo"):
    

[<a class="hero-card-person-photo__item fancybox js-hero-person-card-photo" href="https://cdn.pamyat-naroda.ru/images/roadheroes%2F1c%2F1c00106b1117ef69f73c3271902c7ed9_crop.jpg" rel="hero-card-photo" style="background-image: url(https://cdn.pamyat-naroda.ru/images/roadheroes%2F1c%2F1c00106b1117ef69f73c3271902c7ed9_crop.jpg);"></a>,
 <a class="hero-card-person-photo__item fancybox js-hero-person-card-photo" href="https://cdn.pamyat-naroda.ru/images/roadheroes%2F36%2F3693284544645c2cf94dd8eac0e49320_crop.jpg" rel="hero-card-photo" style="background-image: url(https://cdn.pamyat-naroda.ru/images/roadheroes%2F36%2F3693284544645c2cf94dd8eac0e49320_crop.jpg);"></a>]

In [63]:
soup.find("div", class_="js-hero-card-doc hero-card-docs-item hero-card-docs-item_spisok")

<div class="js-hero-card-doc hero-card-docs-item hero-card-docs-item_spisok" data-box_number="" data-doc-documents-id="" data-doc-id="9205769" data-doc-index="isp" data-doc-pages="[]" data-doc-pages-id="" data-doc-show="N" data-doc-type="chelovek_spisok" data-nomer_dela="1" data-nomer_fonda="6559" data-nomer_opisi="267771с" data-podvig_1="" data-podvig_2="" data-podvig_3="" data-section="ЦАМО" data-shkaf_i_yaschik="" data-shkaf_number="" data-storage_number="">
<div class="hero-card-docs-item__name">
        Пушкин Михаил Константинович    </div>
<div class="hero-card-docs-item__reward">
                                        Именной список части
                    </div>
<div class="heroes_card__doc-box">
<div class="hero-card-docs-item__info" data-dict=" ">
<b>Дата рождения:</b>
                                                                                                                                                
                                                             